[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/littlekg87/lecture/blob/main/2025_kmooc/notebooks/03_word_frequency.ipynb)


# 단어 분석 ① — 김상헌 상소문 단어 빈도 분석

『인조실록』에 실린 김상헌의 상소문에서 **명사만 뽑아 빈도수**를 세어 봅니다.

**왼쪽의 ▶ 버튼을 위에서부터 차례대로 누르기만 하면 됩니다.**

> 💡 **왜 Colab을 쓰나요?**
> 한국어 형태소 분석기 KoNLPy는 내 컴퓨터에 설치하려면 자바(JDK)를 따로 깔아야 해서
> 특히 맥에서 애를 먹습니다. Colab에서는 아래 칸 한 번만 실행하면 끝납니다.


## 0단계 — 준비하기 (설치)


In [ ]:
# 형태소 분석기 KoNLPy 설치 (1~2분 걸립니다)
# KoNLPy는 자바로 만들어져 있어서, 자바(JDK)를 먼저 깔아야 합니다.

!apt-get update -qq
!apt-get install -y -qq openjdk-17-jdk-headless > /dev/null
!pip install -q konlpy

import glob, os

# 설치된 자바 위치를 자동으로 찾아 알려줍니다
jvm = sorted(glob.glob('/usr/lib/jvm/java-*-openjdk-amd64'))
os.environ['JAVA_HOME'] = jvm[-1]
print('JAVA_HOME =', os.environ['JAVA_HOME'])

# 잘 설치됐는지 확인
from konlpy.tag import Okt
okt = Okt()
print(okt.nouns('세종실록 지리지를 분석합니다'))
print('설치 완료!')


## 0단계 — 실습 데이터 내려받기


In [ ]:
# 실습 데이터를 강의 깃허브 저장소에서 곧바로 내려받습니다.
# 내 컴퓨터의 파일 경로를 적을 필요가 없습니다.

GITHUB = "https://raw.githubusercontent.com/littlekg87/lecture/main/2025_kmooc"

FILES = [
    "data/word-analysis/kim-sangheon-sangso.txt",
]

import os
import urllib.request

for path in FILES:
    name = path.split('/')[-1]
    try:
        urllib.request.urlretrieve(f'{GITHUB}/{path}', name)
    except Exception as e:
        raise SystemExit(
            f'내려받기에 실패했습니다: {path}\n'
            f'  · 인터넷 연결을 확인해 주세요.\n'
            f'  · 그래도 안 되면 강의 게시판에 문의해 주세요.\n'
            f'  (원인: {e})'
        )
    print(f'내려받음: {name}  ({os.path.getsize(name):,} 바이트)')

print()
print('준비 완료! 아래 칸부터 차례로 실행하세요.')

# (선택) 깃허브 대신 내 컴퓨터의 파일을 쓰고 싶다면
# 아래 두 줄 앞의 # 을 지우고 실행한 뒤 파일을 선택하세요.
# from google.colab import files
# files.upload()


## 1단계 — 텍스트 파일 읽기


In [ ]:
file_path = 'kim-sangheon-sangso.txt'

with open(file_path, 'r', encoding='utf-8') as file:
    text = file.read()

print(f'글자 수: {len(text):,}자')
print()
print(text[:300])  # 앞부분만 살짝 보기


## 2단계 — 한글 이외의 문자 제거

`[^가-힣\s]` 는 **한글과 공백이 아닌 모든 것**을 뜻합니다.
한자·숫자·문장부호를 지워서 분석 품질을 높입니다.


In [ ]:
import re

text = re.sub(r'[^가-힣\s]', '', text)

print(text[:300])


## 3단계 — 형태소 분석 (명사 추출)


In [ ]:
from konlpy.tag import Okt

okt = Okt()
tokens = okt.nouns(text)

print(f'추출된 명사: {len(tokens):,}개')
print(tokens[:30])


## 4단계 — 불용어(뜻 없는 단어) 제거

결과를 보고 **의미 없는 단어가 상위에 올라오면 아래 목록에 추가**하세요.
이 과정을 두세 번 반복하는 것이 분석의 핵심입니다.


In [ ]:
stopwords = ['것', '저', '그', '이', '수', '있다', '하다']

tokens = [word for word in tokens if word not in stopwords]

# 한 글자 명사는 뜻이 모호한 경우가 많아 함께 걸러 줍니다.
# 한 글자도 보고 싶다면 아래 줄 맨 앞에 # 을 붙이세요.
tokens = [word for word in tokens if len(word) > 1]

print(f'남은 단어: {len(tokens):,}개')


## 5단계 — 단어 빈도수 계산 & 정렬


In [ ]:
from collections import Counter

word_counts = Counter(tokens)
sorted_words = sorted(word_counts.items(), key=lambda x: x[1], reverse=True)

print(f'총 {len(word_counts):,}종류의 단어')
print()
for word, freq in sorted_words[:50]:  # 상위 50개
    print(f'{word}, {freq}회')


## 6단계 — 표로 정리하고 CSV로 내려받기


In [ ]:
import pandas as pd

df = pd.DataFrame(sorted_words, columns=['단어', '빈도'])
df.to_csv('word_frequency.csv', index=False, encoding='utf-8-sig')

display(df.head(20))

from google.colab import files
files.download('word_frequency.csv')


---
다음 실습 → **[단어 분석 ② 워드클라우드](04_wordcloud.ipynb)**
